# Ration Early or Run Dry?

## A reservoir trade-off in four TaqSim blocks

Dr. Tobias Siegfried, hydrosolutions GmbH · CC BY 4.0

<p><img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/hydrosolutions_logo.png" alt="hydrosolutions" width="220"></p>
<p><b>hydrosolutions GmbH</b> · Venusstrasse 29, 8050 Zürich, Switzerland · <a href="https://www.hydrosolutions.ch">www.hydrosolutions.ch</a> · hs@hydrosolutions.ch · +41 43 535 05 80</p>

# Savol

Suv ombori qishda toʻladi. Yozda xoʻjalik undan suv oladi. Koʻp yillarda suv yetarli. Qurgʻoqchil yilda esa
yetmaydi, va operatorning omborni boshqarish uchun ikki yoʻli bor:

- **Ombor boʻshaguncha toʻliq berish.** Xoʻjalik soʻraganini iloji boricha uzoq vaqt toʻliq oladi. Suv tugasa,
  yozning oxirgi haftalarida deyarli hech narsa qolmaydi.
- **Erta cheklash.** Ombor kamayib qolganda xoʻjalikka soʻraganining faqat bir qismini berish. Suv uzoqroqqa
  yetadi, lekin xoʻjalik koʻproq hollarda, hatto suv baribir yetgan boʻladigan yillarda ham, taqchillik koʻradi.

Qaysi biri yaxshiroq? Bu nimani hisoblashingizga bogʻliq. Ushbu daftar buni koʻrsatadigan eng kichik modelni
quradi: toʻrtta TaqSim bloki, ikkita dastak, ikkita maqsad. Soʻngra qidiruvni optimallashtirgichga topshiradi.

**Siz nimani oʻrganasiz**

1. TaqSimda kichik suv tizimini blokma-blok qurish va ishga tushirishni.
2. Nima uchun ikki oqilona maqsad qarama-qarshi tomonga tortishi mumkinligini va barcha maʼqul murosalarni
   birdaniga qanday koʻrishni.
3. Qidiruvni optimallashtirgichga qanday topshirish va u qaytargan natijani qanday oʻqishni.

> **Atamalar.** Muhandislar "ombor boʻshaguncha toʻliq berish"ni *standart boshqaruv siyosati*, "erta
> cheklash"ni esa *xejirlash* (hedging) deb ataydi. Hech bir boshqa reja barcha maqsadlar boʻyicha undan ustun
> kelmaydigan reja *Pareto-optimal*, bunday rejalar toʻplami esa *Pareto fronti* deyiladi. Bu daftarda ular
> "ustun kelib boʻlmaydigan rejalar" deb yuritiladi.

# Toʻrt blokdan iborat tizim

TaqSim suv tizimini **tarmoq** sifatida tasvirlaydi: suvni saqlaydigan, ishlatadigan yoki uzatadigan tugunlar va
ular orasida suv oqadigan qirralar. Bizning tarmoqda toʻrtta tugun va uchta qirra bor.

In [ ]:
# @title Sozlash: avval shu katakni ishga tushiring
# ── Setup: run this cell first. In Colab it takes about a minute. ──────────────────────────────────
# It fetches the course folder, installs TaqSim, and tells Python where the course code lives.
import os
import sys
import subprocess

if not os.path.exists("src/zarafshan_taqsim"):  # not inside the course folder yet (the normal case in Colab)
    if not os.path.exists("zarafshan-taqsim-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/hydrosolutions/zarafshan-taqsim-course.git"], check=True)
    os.chdir("zarafshan-taqsim-course")
try:
    import taqsim  # noqa: F401  (already installed when you work on your own computer)
except ImportError:
    %pip install -q deap ctrl-freak pymoo
    %pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
sys.path[:0] = [os.path.abspath("src")]
print("Ready. Working folder:", os.getcwd())


In [ ]:
# @title Kod: setup
#| label: setup
# Standard tools
import inspect  # to show the source code of a function or class inside this notebook
import logging

import pandas as pd  # tables
from IPython.display import HTML, Markdown  # to show formatted text and HTML as cell output

# TaqSim itself: the four kinds of node we use, the edge that joins them, and the system that holds them
from taqsim import Demand, Edge, Sink, Source, Storage, TimeSeries, WaterSystem
from taqsim.node import NoLoss  # a loss rule that loses nothing (reservoirs need one)
from taqsim.node.events import DeficitRecorded  # the event a Demand node records when it is short
from taqsim.time import Frequency  # the length of one simulation step (monthly here)

# The helpers written for this coursebook (src/zarafshan_taqsim/rationing_coursebook.py)
from zarafshan_taqsim.rationing_coursebook import (
    CAPACITY_MM3,
    EVAPORATION_SHARE_EXAMPLE,
    MONTHLY_NEED_MM3,
    NO_RATIONING,
    PAPER_REGIME,
    START_STORAGE_MM3,
    YEARS,
    Plan,
    RationingRelease,
    SUMMER_PLAN_FOR_WINTER,
    WINTER_FIELDS_TOTAL_MM3,
    WINTER_IN_FULL,
    SummerEvaporation,
    WinterPlan,
    WinterRelease,
    drought_window_start,
    explorer_html,
    explorer_payload,
    front_of,
    objectives,
    optimise_plans,
    plan_grid,
    forecast_value,
    least_winter_shortage,
    plot_forecast_skill,
    plot_forecast_value,
    plot_front_shift,
    plot_inflows,
    plot_network,
    plot_regime_comparison,
    plot_runs,
    plot_score_maps,
    plot_system_sketch,
    plot_tradeoff,
    plot_winter_decisions,
    plot_winter_fronts,
    score_by_skill,
    score_plans,
    score_winter_plans,
    season_statistics,
    seasonal_forecast,
    simulate_plan,
    simulate_winter,
    synthetic_inflows,
    winter_grid,
    winter_plan_within,
)

logging.basicConfig(level=logging.WARNING)  # only show warnings and errors, not progress messages
pd.set_option("display.precision", 2)  # two decimals in tables

LANG = "uz"  # the web page shows all three languages and lets you switch; a notebook is built for one language


def say(en: str, uz: str, ru: str) -> Markdown:
    """Text in three languages: all three on the web page (the switcher picks one), one in a notebook."""
    if LANG:
        return Markdown({"en": en, "uz": uz, "ru": ru}[LANG])
    blocks = (("en", en), ("uz", uz), ("ru", ru))
    return Markdown("\n\n".join(f"::: {{.lang-{code}}}\n{text}\n:::" for code, text in blocks))


def show_source(thing) -> Markdown:
    """Show the source code of a function or class as a code block."""
    return Markdown(f"```python\n{inspect.getsource(thing)}```")

In [ ]:
# @title Kod: fig-network
#| label: fig-network
#| fig-cap: "The model as a network. Each node is one TaqSim block; each edge is named after the nodes it joins."
#| echo: false
_ = plot_network()

- **river** (daryo) — `Source`: har oy sonlar roʻyxatidan tarmoqqa suv kiritadi.
- **reservoir** (suv ombori) — `Storage`: suvni sigʻimigacha saqlaydi, bir qismini yoʻqotadi (agar shunday
  buyursak) va bir qismini qoida boʻyicha chiqaradi. Qarorlar shu yerda qabul qilinadi.
- **farm** (xoʻjalik) — `Demand`: har oy maʼlum miqdorni soʻraydi va qanchasini olmaganini yozib boradi.
- **downstream** (quyi oqim) — `Sink`: xoʻjalik ishlatmagan suv tarmoqni shu yerdan tark etadi.

In [ ]:
# @title Kod: fig-system
#| label: fig-system
#| fig-cap: "The same four blocks with their numbers. Water moves from left to right."
#| echo: false
_ = plot_system_sketch()

Raqamlar suv omborlari samaradorligi haqidagi klassik maqoladan olingan (Hashimoto, Stedinger va Loucks, 1982,
17-bet va 1-jadval), bitta oʻzgarish bilan. Maqoladagi daryo qishda toʻlib oqadi. Markaziy Osiyo daryolari esa qor
va muzliklar erishidan toʻyinadi: ular yozda toʻlib oqadi, qishda esa eng kam suvli. Shuning uchun biz maqoladagi
ikki faslni oʻrin almashtirdik, qolgan hamma narsani saqlab qoldik. [Taqqoslash boʻlimi](#sec-paper-regime)
maqoladagi daryoni biznikining yonida hisoblaydi. Hajmlar million kub metrda (mln m³).

| Miqdor | Qiymat |
|---|---|
| Suv ombori sigʻimi | 40 mln m³ |
| Xoʻjalikka yozda kerak | 45 mln m³ (olti oy davomida oyiga 7,5 mln m³) |
| Xoʻjalikka qishda kerak boʻlgan suv | 5 mln m³ (olti oy davomida oyiga 0,83 mln m³): mavsumdan tashqari kichik talab, maqoladan olingan |
| Daryo oqimi qishda | oʻrtacha 25 mln m³, standart chetlanish 10 mln m³ (maqolada: 40 va 15) |
| Daryo oqimi yozda | oʻrtacha 40 mln m³, standart chetlanish 15 mln m³ (maqolada: 25 va 10) |

Yozgi ehtiyoj (45) oʻrtacha yozgi oqimdan (40) katta, shuning uchun xoʻjalik qishda saqlangan suvga bogʻliq. Suv
ombori (40) ehtiyojdan kichik va uni faqat kam suvli qishki oqimdan toʻldirish mumkin, shu bois xatoga oʻrin kam.

Model yili — oltita qishki oy, soʻngra oltita yozgi oy; kalendar sanalari biriktirilmagan. Uni Zarafshon
havzasida qoʻllaniladigan gidrologik yil deb oʻqing: qish — oktyabrdan martgacha, yoz (sugʻorish mavsumi) —
apreldan sentyabrgacha. Mavsum ichida oydan oyga hech narsa oʻzgarmaydi: har bir mavsumning oqimi oltita oyga
teng taqsimlanadi, xoʻjalikning talabi ham shunday. Yagona mavsumiylik — maqoladagi kabi qish va yoz orasidagi
farq. Zarafshonning iyuldagi choʻqqisi bilan haqiqiy oylik shakli oʻn olti blokli oʻyinda va modellashtirish
darsligida koʻrsatilgan.

## Daryo

Bizda yuz yillik oʻlchovlar yoʻq, shuning uchun daryoni oʻzimiz yaratamiz: yuqoridagi jadvaldagi oʻrtacha qiymatlar
va tarqalish bilan yuz yillik qishki va yozgi oqimlar. Haqiqiy daryolardagidek, qurgʻoqchil fasldan keyin koʻpincha
yana qurgʻoqchil fasl keladi.

In [ ]:
# @title Kod: river
#| label: river
inflows = synthetic_inflows()  # 100 years x 12 months = 1 200 numbers, each a month's inflow in Mm³

# Check the river against the numbers we asked for: averages, spread and the link from one season to the next
season_statistics(inflows).style.hide(axis="index").format(precision=2)

`synthetic_inflows()` har doim bir xil daryoni beradi (u qatʼiy tasodifiy urugʻdan foydalanadi), shuning uchun bu
daftardagi har bir natijani takrorlash mumkin. Jadval daryoning statistikasini biz soʻragan qiymatlar bilan
taqqoslaydi; ular bir necha foiz aniqlikda mos keladi.

In [ ]:
# @title Kod: fig-river
#| label: fig-river
#| fig-cap: "One hundred synthetic years. In many years the summer inflow is far below what the farm needs."
#| echo: false
_ = plot_inflows(inflows)

> **Maqoladan farqlar.** (1) Yuqorida tushuntirilganidek, fasllar oʻrin almashtirilgan. (2) Har bir faslning oqimi
> oltita oylik qadamga teng taqsimlangan. (3) Maqola boshqaruv qoidalarini matematik yoʻl bilan keltirib chiqaradi;
> biz ikki dastakli oddiy qoidadan foydalanamiz. (4) Fasllar orasidagi bogʻliqlik oqimlarning logarifmlariga
> qoʻllangan, bu — taxmin. Shu bois bu daftardagi raqamlar bizniki, maqoladagilarning takrori emas.

# TaqSimda qurish

## Ikki dastakli qoida

**Klass nima, oddiy soʻzlar bilan.** TaqSim kabi dastur *obyektlardan* qurilgan: suv ombori, xoʻjalik, qoida.
**Klass** bir turdagi obyektni tasvirlaydi: u nimani eslab qoladi va nima qila oladi. Bir nechta toʻldiriladigan
katagi va orqasida qisqa koʻrsatmasi bor bosma blankni tasavvur qiling. Klass — boʻsh blank; undan yasalgan har bir
obyekt — toʻldirilgan bitta nusxa. Bizning qoida klassimiz ikkita raqamni eslab qoladi, chegara va cheklash (uning
*maydonlari*), va bitta ishni qilishni biladi, `release(...)`, TaqSim buni har oy undan soʻraydi (uning *metodi*).
`RationingRelease(trigger=16, ration=0.8)` deb yozish blankning bitta nusxasini toʻldiradi. Quyidagi `Plan`,
`Storage` va `Demand` — oʻsha turdagi boshqa blanklar. Bu yerda "obyektga yoʻnaltirilgan" degani shu, xolos: model —
bir-biriga suv uzatadigan bir nechta shunday blank. Bu daftarni oʻqish yoki oʻzgartirish uchun boshqa hech narsa
kerak emas.

TaqSim suv omboriga *chiqarish qoidasi* kerak: har oy qancha suv chiqarishni aytadigan kichik klass. Bizning
qoidada ikkita dastak bor.

- **Chegara** (trigger, mln m³): omborda suv shundan kam boʻlganda cheklash boshlanadi.
- **Ulush** (ration, 0 dan 1 gacha): shunda yozgi ehtiyojning qancha qismi beriladi.

In [ ]:
# @title Kod: rule-source
#| label: rule-source
#| echo: false
show_source(RationingRelease)

Klassni yuqoridan oʻqiymiz:

- `@dataclass(frozen=True)` uni kichik, oʻzgarmas yozuvga aylantiradi: unda ikki dastakdan boshqa hech narsa yoʻq.
- `__params__` optimallashtirgich burashi mumkin boʻlgan dastaklarni nomlaydi, `__bounds__` esa qaysi qiymatlar
  orasida ekanini aytadi.
- `trigger` va `ration` — dastaklar, standart qiymatlari bilan (0 va 1: cheklash yoʻq).
- `release(...)` ni TaqSim har oy bir marta chaqiradi. U suv omborini (`node`, undan joriy suv hajmini oʻqiydi),
  shu oyning oqimini va `t` vaqt qadamini oladi va chiqariladigan hajmni mln m³ da qaytaradi.

Chegara 0 (yoki ulush 1) — umuman cheklash yoʻq degani.

## Toʻrt blokni ulash

TaqSim modelini qurish har doim bir xil toʻrt qadamdan iborat: boʻsh tizim yaratish, tugunlarni qoʻshish,
qirralarni qoʻshish, tekshirish va ishga tushirish.

In [ ]:
# @title Kod: wire
#| label: wire
# 1. An empty system. One simulation step is one month.
system = WaterSystem(frequency=Frequency.MONTHLY)

# 2. The four nodes. Each has an id (its name in the network) and the settings its kind needs.
system.add_node(
    Source(
        id="river",
        inflow=TimeSeries(values=list(inflows)),  # one value per month, Mm³
    )
)
system.add_node(
    Storage(
        id="reservoir",
        capacity=CAPACITY_MM3,  # 40 Mm³; more than this spills
        initial_storage=START_STORAGE_MM3,  # 20 Mm³ in store when the simulation starts
        release_policy=RationingRelease(trigger=0.0, ration=1.0),  # our rule; these knobs mean no rationing
        loss_rule=NoLoss(),  # no evaporation for now
    )
)
system.add_node(
    Demand(
        id="farm",
        requirement=TimeSeries(values=list(MONTHLY_NEED_MM3) * YEARS),  # the 12 monthly needs, repeated 100 times
    )
)
system.add_node(Sink(id="downstream"))

# 3. The three edges. An edge is named after the two nodes it joins and carries whatever its source lets out.
for source, target in [("river", "reservoir"), ("reservoir", "farm"), ("farm", "downstream")]:
    system.add_edge(Edge(id=f"{source}_to_{target}", source=source, target=target))

# 4. Check that the network is complete (every node connected, every setting valid), then run all 1 200 months.
system.validate()
system.simulate(len(inflows))

Tizimni ishga tushirish hech narsa chop etmaydi. Buning oʻrniga har bir tugun **hodisalar** yozuvini yuritadi:
oyma-oy nima qabul qilgani, saqlagani, chiqargani, yoʻqotgani yoki yetkazib bera olmagani. Xoʻjalik kerakligidan
kam olgan har bir oyda `DeficitRecorded` hodisasini yozadi, va har bir bunday hodisa yetishmagan hajmni oʻz ichida
saqlaydi:

In [ ]:
# @title Kod: first-read
#| label: first-read
farm = system.nodes["farm"]  # look up a node by its id

# Add up the missing volume over all deficit events, then divide by the number of years
not_delivered = sum(event.deficit for event in farm.events_of_type(DeficitRecorded))
print(f"Water not delivered: {not_delivered / YEARS:.2f} Mm³ per year, on average")

Bundan keyin `simulate_plan(inflows, plan)` aynan shu qadamlarni bajaradi (qurish, tekshirish, hisoblash,
hodisalarni oʻqish) va natijalarni tartibli koʻrinishda qaytaradi: oyma-oy suv hajmi, oyma-oy taqchillik va keyingi
boʻlimda kiritiladigan uchta koʻrsatkich.

# Birinchi hisob: ombor boʻshaguncha toʻliq berish

In [ ]:
# @title Kod: fig-first-run
#| label: fig-first-run
#| fig-cap: "No rationing. Left: water in the reservoir over twenty of the hundred years. Right: for each year, how short the worst month was."
no_rationing = simulate_plan(inflows, NO_RATIONING)  # NO_RATIONING is Plan(trigger=0, ration=1)
first_year = drought_window_start(no_rationing)  # a 20-year window that contains the worst drought

# One row per plan; the right panel shows the 20 years starting in first_year, shaded in the left panel
_ = plot_runs({"Deliver in full until empty": no_rationing}, first_year=first_year)

Chap panelda barcha 100 yil koʻrsatilgan. Har bir yil uchun bitta ustun: oʻsha yilning eng yomon oyi, yaʼni
xoʻjalik oʻsha oy talabining qancha ulushini olmagani. 72 % lik ustun — eng yomon oyda xoʻjalik soʻraganining
faqat 28 % ini olgan degani; koʻp yillarda umuman ustun yoʻq. Oʻng panel eng qattiq qurgʻoqchilik atrofidagi
belgilangan 20 yilni yaqindan koʻrsatadi: suv ombori har qish toʻladi (koʻtarilayotgan chiziqlar) va har yoz
boʻshatiladi, eng qurgʻoqchil yillarda esa nolga yetadi.

Rejani uchta raqam bilan baholaymiz:

- **Oʻrtacha taqchillik**: xoʻjalikka kerak boʻlgan, lekin olmagan suv, yiliga mln m³.
- **Eng yomon oy**: yuz yil ichidagi eng yomon oyda xoʻjalik ehtiyojining olmagan ulushi.
- **Taqchil yillar**: kamida bitta taqchil oyi boʻlgan yillar soni.

In [ ]:
# @title Kod: first-scores
#| label: first-scores
#| echo: false
s = no_rationing.scores
say(
    f"Without rationing the farm is short in **{s.years_short} of 100 years**. On average **{s.mean_shortage:.1f} Mm³ "
    f"per year** is not delivered, which is little. But when it goes wrong, it goes badly wrong: in the worst month "
    f"the farm gets **{100 - s.worst_month:.0f} %** of what it needs ({s.worst_month:.0f} % short).",
    f"Cheklashsiz xoʻjalik **100 yildan {s.years_short} tasida** taqchillik koʻradi. Oʻrtacha **yiliga "
    f"{s.mean_shortage:.1f} mln m³** yetkazib berilmaydi, bu kam. Lekin ishkal chiqqanda, qattiq chiqadi: eng yomon "
    f"oyda xoʻjalik keragining **{100 - s.worst_month:.0f} %** ini oladi ({s.worst_month:.0f} % taqchillik).",
    f"Без лимитирования хозяйство испытывает дефицит в **{s.years_short} из 100 лет**. В среднем не подаётся "
    f"**{s.mean_shortage:.1f} млн м³ в год** — немного. Но когда дело идёт плохо, оно идёт очень плохо: в худшем "
    f"месяце хозяйство получает **{100 - s.worst_month:.0f} %** потребности (дефицит {s.worst_month:.0f} %).",
)

# Ikkinchi hisob: cheklash

Ikkita cheklash rejasi. Ikkalasi ham omborda 16 mln m³ dan kam suv qolganda cheklashni boshlaydi. Biri suv
berishni ozgina kamaytiradi (80 % gacha), ikkinchisi keskin kesadi (50 % gacha).

In [ ]:
# @title Kod: fig-second-run
#| label: fig-second-run
#| fig-cap: "Three plans, same river. The grey line marks the trigger."
# A Plan holds the two knobs. Below 16 Mm³ in store, deliver 80 % (lightly) or 50 % (deep) of the summer need.
ration_lightly = simulate_plan(inflows, Plan(trigger=16, ration=0.8))
cut_deep = simulate_plan(inflows, Plan(trigger=16, ration=0.5))

# A dictionary of name -> run; the names become the panel titles
runs = {
    "Deliver in full until empty": no_rationing,
    "Ration lightly (below 16 Mm³, deliver 80 %)": ration_lightly,
    "Cut deep (below 16 Mm³, deliver 50 %)": cut_deep,
}
_ = plot_runs(runs, first_year=first_year)

Chapdagi ustunlarga qarang. "Ozgina cheklash" bir nechta baland ustunni 20 % lik koʻplab past ustunlarga
aylantiradi, lekin qurgʻoqchilikdagi eng baland ustun qoladi. "Keskin kesish" ostida hech bir ustun 50 % dan baland
emas: eng yomon holat — cheklashning oʻzi, chunki suv ombori hech qachon butunlay boʻshamaydi. Buning bahosi
qolgan ikki koʻrsatkichda koʻrinadi.

In [ ]:
# @title Kod: tbl-three-plans
#| label: tbl-three-plans
#| tbl-cap: "The three plans, scored."
#| echo: false
pd.DataFrame(
    [
        (name, run.scores.mean_shortage, run.scores.worst_month, run.scores.years_short)
        for name, run in runs.items()
    ],
    columns=["Plan", "Average shortage (Mm³/yr)", "Worst month (% short)", "Years short (of 100)"],
).style.hide(axis="index").format({"Average shortage (Mm³/yr)": "{:.1f}", "Worst month (% short)": "{:.0f}"})

In [ ]:
# @title Kod: second-scores
#| label: second-scores
#| echo: false
light, deep = ration_lightly.scores, cut_deep.scores
deep_min = cut_deep.storage.min()  # the lowest the reservoir ever gets under the deep cut
say(
    f"- **How to read the storage panel.** The grey line is the trigger, 16 Mm³. Every summer month in which the "
    f"blue line is below it is a rationed month. Without rationing the line hits zero in the drought; under "
    f"\"Cut deep\" it never falls below {deep_min:.1f} Mm³, because halving the deliveries keeps water back.\n"
    f"- **How to read the bars.** Under \"Ration lightly\" most bars are 20 % tall: that is the ration itself, "
    f"in every year the trigger is reached. There are more of them ({light.years_short} years instead of "
    f"{s.years_short}), and the tallest one, {light.worst_month:.0f} %, is still there. Under \"Cut deep\" the bars "
    f"are 50 % at most and there are {deep.years_short} of them.\n"
    f"- **Rationing lightly does not help here.** The worst month is still {light.worst_month:.0f} % short: in the "
    f"worst drought the reservoir runs dry anyway. The farm is simply short more often "
    f"({light.years_short} years instead of {s.years_short}).\n"
    f"- **Cutting deep works, at a price.** The worst month improves from {s.worst_month:.0f} % to "
    f"{deep.worst_month:.0f} % short. But {deep.mean_shortage:.1f} Mm³ per year is not delivered instead of "
    f"{s.mean_shortage:.1f}, and the farm is short in {deep.years_short} years instead of {s.years_short}.\n\n"
    f"So there is no free lunch: a milder worst month costs water. How much, exactly?",
    f"- **Suv hajmi panelini qanday oʻqish.** Kulrang chiziq — chegara, 16 mln m³. Koʻk chiziq undan pastda boʻlgan "
    f"har bir yoz oyi — cheklangan oy. Cheklashsiz chiziq qurgʻoqchilikda nolga tushadi; \"Keskin kesish\" ostida "
    f"u hech qachon {deep_min:.1f} mln m³ dan pastga tushmaydi, chunki suv berishni yarmiga kamaytirish suvni saqlab "
    f"qoladi.\n"
    f"- **Ustunlarni qanday oʻqish.** \"Ozgina cheklash\" ostida koʻp ustunlar 20 % balandlikda: bu cheklashning "
    f"oʻzi, chegaraga yetilgan har bir yilda. Ular koʻproq ({s.years_short} oʻrniga {light.years_short} yil), eng "
    f"balandi esa, {light.worst_month:.0f} %, hali ham turibdi. \"Keskin kesish\" ostida ustunlar koʻpi bilan 50 % va "
    f"ular {deep.years_short} ta.\n"
    f"- **Ozgina cheklash bu yerda yordam bermaydi.** Eng yomon oy hali ham {light.worst_month:.0f} % taqchil: eng "
    f"qattiq qurgʻoqchilikda ombor baribir quriydi. Xoʻjalik shunchaki koʻproq taqchillik koʻradi "
    f"({s.years_short} oʻrniga {light.years_short} yil).\n"
    f"- **Keskin kesish ishlaydi, lekin bahosi bor.** Eng yomon oy {s.worst_month:.0f} % dan {deep.worst_month:.0f} % "
    f"taqchillikka yaxshilanadi. Lekin {s.mean_shortage:.1f} oʻrniga yiliga {deep.mean_shortage:.1f} mln m³ "
    f"yetkazilmaydi, va xoʻjalik {s.years_short} oʻrniga {deep.years_short} yilda taqchillik koʻradi.\n\n"
    f"Demak, tekin tushlik yoʻq: yumshoqroq eng yomon oy suvga tushadi. Aniq qanchaga?",
    f"- **Как читать панель с запасом.** Серая линия — порог, 16 млн м³. Каждый летний месяц, в который синяя "
    f"линия ниже порога, — месяц с лимитированием. Без лимитирования линия в засуху доходит до нуля; при "
    f"«сильном урезании» она никогда не опускается ниже {deep_min:.1f} млн м³, потому что половинная подача "
    f"сберегает воду.\n"
    f"- **Как читать столбики.** При «лёгком лимитировании» большинство столбиков высотой 20 %: это само "
    f"урезание, в каждый год, когда достигнут порог. Их больше ({light.years_short} лет вместо {s.years_short}), а "
    f"самый высокий, {light.worst_month:.0f} %, никуда не делся. При «сильном урезании» столбики не выше 50 %, и их "
    f"{deep.years_short}.\n"
    f"- **Лёгкое лимитирование здесь не помогает.** Худший месяц по-прежнему с дефицитом {light.worst_month:.0f} %: "
    f"в самую сильную засуху водохранилище всё равно пустеет. Хозяйство просто чаще испытывает дефицит "
    f"({light.years_short} лет вместо {s.years_short}).\n"
    f"- **Сильное урезание работает, но имеет цену.** Худший месяц улучшается с {s.worst_month:.0f} % до "
    f"{deep.worst_month:.0f} % дефицита. Но не подаётся {deep.mean_shortage:.1f} млн м³ в год вместо "
    f"{s.mean_shortage:.1f}, и хозяйство испытывает дефицит в {deep.years_short} годах вместо {s.years_short}.\n\n"
    f"Бесплатных обедов нет: более мягкий худший месяц стоит воды. Сколько именно?",
)

## Oʻz rejangizni sinab koʻring

In [ ]:
# @title Kod: fig-my-plan
#| label: fig-my-plan
#| fig-cap: "Your plan against no rationing."
my_plan = Plan(trigger=22, ration=0.55)  # change these two numbers and run the cell again

mine = simulate_plan(inflows, my_plan)
_ = plot_runs({"Deliver in full until empty": no_rationing, "My plan": mine}, first_year=first_year)
print(mine.scores)  # the three scores of your plan

Chegara 0 dan 40 mln m³ gacha (sigʻim), ulush 0 dan 1 gacha har qanday qiymat boʻlishi mumkin. Chegara 40 — "yozda
har doim chekla" degani.

Nimaga qarash kerak: ikkala qator bir xil oʻqlarga ega, shuning uchun ularni toʻgʻridan-toʻgʻri solishtiring.
Rejangizning uchta koʻrsatkichi rasm ostida chop etiladi. Reja faqat bitta koʻrsatkichni qolgan ikkitasini
yomonlashtirmasdan yaxshilasagina yaxshiroq; koʻp oʻzgarishlar birini ikkinchisiga almashtiradi. Ikkita foydali
chegara: chegara 0 — "hech qachon cheklamaslik" (yana birinchi qator), chegara 40 — "har bir yoz oyida cheklash".

# Barcha rejalar birdaniga

> **Masala formulalarda.** Oylar $N = 100$ yil boʻylab $t = 1, \dots, 12N$ deb raqamlanadi; $m(t)$ — yil ichidagi
> oy, $S$ — oltita yoz oyi. $t$ oy boshida omborda $s_t$ suv bor va unga $q_t$ oqim keladi; $K = 40$ mln m³
> sigʻimdan ortiqchasi toshib ketadi, $\ell_t$ yoʻqotish (bugʻlanish boʻlimigacha nol) ayiriladi. Qoida koʻradigan
> suv — $\hat s_t$. Reja — $\theta = (\tau, \rho)$ juftligi, chegara va ulush; xoʻjalik talabi $d_m$ (qish oyida
> 0,83 mln m³, yoz oyida 7,5). Chiqarish va suv balansi:
>
$$
r_t(\theta) = \min\{\hat s_t,\; \rho_t\, d_{m(t)}\}, \qquad
\rho_t = \begin{cases} \rho & \text{if } m(t) \in S \text{ and } \hat s_t < \tau \\ 1 & \text{otherwise,} \end{cases}
$$
>
$$
\hat s_t = \min\{K,\; s_t + q_t\} - \ell_t, \qquad s_{t+1} = \hat s_t - r_t,
$$
>
> $t$ oyning taqchilligi — $\delta_t = d_{m(t)} - r_t$. Ikki koʻrsatkich, yiliga mln m³ da va foizda:
>
$$
f_1(\theta) = \frac{1}{N}\sum_{t=1}^{12N} \delta_t(\theta), \qquad
f_2(\theta) = 100 \max_t \frac{\delta_t(\theta)}{d_{m(t)}},
$$
>
> masala esa ikkalasini birdaniga kichik qilish:
>
$$
\min_{\theta \in [0,K] \times [0,1]} \big(f_1(\theta),\, f_2(\theta)\big),
$$
>
> Ikki son bir vaqtda eng kichik boʻla olmaydi, shuning uchun "yechish" degani — boshqa hech bir reja ustun kela
> olmaydigan rejalarni topish. $\theta$ reja yengilmagan, agar $f_1(\theta') \le f_1(\theta)$ va
> $f_2(\theta') \le f_2(\theta)$ (biri qatʼiy) boʻlgan $\theta'$ mavjud boʻlmasa; yengilmagan rejalar toʻplami —
> Pareto fronti. Toʻr $f$ ni $\theta$ ning 21 × 21 qiymatida hisoblaydi; optimallashtirgich butun toʻrtburchakni
> qidiradi.

Ikki dastak uchun hammasini shunchaki sinab koʻrish mumkin: 21 chegara × 21 ulush = 441 reja, 441 ta TaqSim
hisobi.

In [ ]:
# @title Kod: grid
#| label: grid
plans = plan_grid()  # 441 Plans: triggers 0, 2, 4, ... 40 Mm³ combined with rations 0, 0.05, 0.10, ... 1
table = score_plans(inflows, plans)  # runs each plan through TaqSim; one row per plan with its three scores

table.sample(5, random_state=1).sort_index().style.hide(axis="index").format(precision=2)  # five rows, as a taste

Har bir qator — bitta reja: ikki dastagi, uchta koʻrsatkichi va quyida tushuntiriladigan `unbeaten` belgisi.
Barcha 441 hisob bir necha soniya oladi.

Keyingi rasm barcha 441 rejani uchta xaritada koʻrsatadi, har bir koʻrsatkich uchun bittadan. Har bir xarita —
shaxmat taxtasi: chegara — koʻndalangiga, cheklash — yuqoriga, har bir katak bitta reja, qoramtiri yomonroq. Chap
xarita: qanchalik oʻngga va pastga (erta cheklash, kam berish), shunchalik koʻp suv ushlab qolinadi va hech qachon
yetkazilmaydi. Oʻrta xarita: eng och kataklar, eng yumshoq eng yomon oylar, yuqori oʻngda — baland chegara va 60 dan
75 % gacha cheklash; pastdagi qora yoʻl — "cheklash boshlangach deyarli hech narsa bermaslik". Oʻng xarita: taqchil
yillar soni chegara bilan oʻsadi, chunki cheklash koʻproq yilda yoqiladi. Hech bir katak uchala xaritada ham och
emas. Murosa shu.

In [ ]:
# @title Kod: fig-maps
#| label: fig-maps
#| fig-cap: "Each score for every plan. Darker is worse. The three maps do not agree on where the good plans are."
#| echo: false
_ = plot_score_maps(table)

Endi ikkita koʻrsatkichni bir-biriga qarshi qoʻyamiz. Har bir reja bitta nuqtaga aylanadi.

In [ ]:
# @title Kod: fig-tradeoff
#| label: fig-tradeoff
#| fig-cap: "All 441 plans. Left is better (less water undelivered). Down is better (milder worst month). A plan is 'beaten' if another plan is at least as good on both counts and better on one."
# Every row of the table becomes a dot; `named` runs are circled and labelled
_ = plot_tradeoff(table, named={"deliver in full": no_rationing, "ration lightly": ration_lightly, "cut deep": cut_deep})

Qora nuqtalar — **ustun kelib boʻlmaydigan rejalar**. Ularning har biri uchun eng yomon oyni yumshatishning yagona
yoʻli — koʻproq suvni yetkazmay qoldirish. Muhokamaga arziydigan rejalar faqat shular; qaysi birini tanlash —
hisob emas, mulohaza.

In [ ]:
# @title Kod: tradeoff-howto
#| label: tradeoff-howto
#| echo: false
front = front_of(table)
head, tail = front.iloc[0], front.iloc[-1]
say(
    f"How to read the chart: one dot per plan. Across, the average shortage, less is better; up, the worst month, "
    f"lower is better. The best corner is the bottom left, and no plan reaches it. The circled dots are the three "
    f"plans of the earlier sections: \"ration lightly\" sits almost on top of \"deliver in full\", \"cut deep\" is "
    f"on the staircase. The staircase joins the unbeaten plans. \"Plans lie further right\" means the axis is cut "
    f"off so that the interesting part stays readable.\n\n"
    f"The table below lists the staircase from its top-left end to its bottom-right end: from no rationing "
    f"({head['mean_shortage']:.2f} Mm³ per year undelivered, worst month {head['worst_month']:.0f} % short, "
    f"{head['years_short']:.0f} years short) to the deepest plan still worth having (trigger {tail['trigger']:.0f} Mm³, "
    f"ration {tail['ration']:.2f}: {tail['mean_shortage']:.2f} Mm³ per year, {tail['worst_month']:.0f} %, "
    f"{tail['years_short']:.0f} years short). Every step down the table buys a milder worst month with more water "
    f"left undelivered and more years with some shortage.",
    f"Diagrammani qanday oʻqish: har bir nuqta bitta reja. Koʻndalangiga — oʻrtacha taqchillik, kami yaxshi; yuqoriga "
    f"— eng yomon oy, pasti yaxshi. Eng yaxshi burchak — pastki chap, va hech bir reja unga yetmaydi. Doira ichidagi "
    f"nuqtalar — oldingi boʻlimlardagi uchta reja: \"ozgina cheklash\" deyarli \"toʻliq berish\" ustida turibdi, "
    f"\"keskin kesish\" zinapoyada. Zinapoya yengilmagan rejalarni birlashtiradi. \"Rejalar oʻngroqda\" degani — "
    f"qiziqarli qism oʻqilishi uchun oʻq kesilgan.\n\n"
    f"Quyidagi jadval zinapoyani yuqori chap uchidan pastki oʻng uchigacha sanab oʻtadi: cheklashsizdan (yiliga "
    f"{head['mean_shortage']:.2f} mln m³ yetkazilmagan, eng yomon oy {head['worst_month']:.0f} % taqchil, "
    f"{head['years_short']:.0f} taqchil yil) hali ham arziydigan eng chuqur rejagacha (chegara {tail['trigger']:.0f} "
    f"mln m³, cheklash {tail['ration']:.2f}: yiliga {tail['mean_shortage']:.2f} mln m³, {tail['worst_month']:.0f} %, "
    f"{tail['years_short']:.0f} taqchil yil). Jadvalda har bir qadam pastga yumshoqroq eng yomon oyni koʻproq "
    f"yetkazilmagan suv va koʻproq taqchil yil evaziga sotib oladi.",
    f"Как читать график: одна точка — один план. По горизонтали средний дефицит, меньше — лучше; по вертикали худший "
    f"месяц, ниже — лучше. Лучший угол — левый нижний, и ни один план до него не дотягивает. Обведённые точки — три "
    f"плана из предыдущих разделов: «лёгкое лимитирование» лежит почти поверх «подавать полностью», «сильное урезание» "
    f"— на лестнице. Лестница соединяет непревзойдённые планы. «Планы лежат правее» означает, что ось обрезана, чтобы "
    f"интересная часть оставалась читаемой.\n\n"
    f"Таблица ниже перечисляет лестницу от её левого верхнего конца до правого нижнего: от отсутствия лимитирования "
    f"({head['mean_shortage']:.2f} млн м³ в год не подано, худший месяц {head['worst_month']:.0f} % дефицита, "
    f"{head['years_short']:.0f} дефицитных лет) до самого глубокого плана, который ещё имеет смысл (порог "
    f"{tail['trigger']:.0f} млн м³, подача {tail['ration']:.2f}: {tail['mean_shortage']:.2f} млн м³ в год, "
    f"{tail['worst_month']:.0f} %, {tail['years_short']:.0f} дефицитных лет). Каждый шаг вниз по таблице покупает "
    f"более мягкий худший месяц ценой большего объёма неподанной воды и большего числа лет с дефицитом.",
)

In [ ]:
# @title Kod: tbl-front
#| label: tbl-front
#| tbl-cap: "The unbeaten plans, from least water undelivered to mildest worst month."
#| echo: false
front = front_of(table)
front[["trigger", "ration", "mean_shortage", "worst_month", "years_short"]].rename(
    columns={
        "trigger": "Trigger (Mm³)",
        "ration": "Ration",
        "mean_shortage": "Average shortage (Mm³/yr)",
        "worst_month": "Worst month (% short)",
        "years_short": "Years short (of 100)",
    }
).style.hide(axis="index").format(
    {"Trigger (Mm³)": "{:.0f}", "Ration": "{:.2f}", "Average shortage (Mm³/yr)": "{:.2f}", "Worst month (% short)": "{:.0f}"}
)

# Qidiruvni optimallashtirgichga topshiramiz

Hammasini sinab koʻrish faqat ikki dastak boʻlgani uchun ishladi. Haqiqiy suv ombori qoidasida oʻnlab dastak bor
(har oy uchun alohida chegara va ulush, bir nechta ombor, bir nechta suv isteʼmolchisi). Oʻnta dastak va har
birida 21 qiymat bilan "hammasi" — hech kim kutib oʻtira olmaydigan darajada koʻp hisob.

Buning oʻrniga optimallashtirgich qidiradi (TaqSim NSGA-II deb ataladigan usuldan foydalanadi). U tasodifiy
rejalardan boshlaydi, ustun kelib boʻlmaydiganlarini saqlab qoladi, ularni aralashtirib va ozgina oʻzgartirib yangi
rejalar yasaydi, va buni takrorlaydi. Biz unga tizimni va ikki maqsadni beramiz:

In [ ]:
# @title Kod: objectives-source
#| label: objectives-source
#| echo: false
show_source(objectives)

Har bir `Objective` da nom, yoʻnalish va hisoblangan tizimni bitta raqamga aylantiradigan funksiya bor.

In [ ]:
# @title Kod: optimise
#| label: optimise
found = optimise_plans(
    inflows,
    pop_size=40,  # the optimizer works with 40 plans at a time
    generations=40,  # and improves them 40 times: 1 600 TaqSim runs in all
    seed=42,  # the starting plans are random; a fixed seed makes the result reproducible
)
print(f"The optimizer returned {len(found)} plans, {len(front_of(found))} of them different.")

Ichkarida `optimise_plans` — TaqSimga bitta chaqiruv, `optimize(system, objectives, timesteps, pop_size,
generations, seed)`, soʻngra qaytarilgan har bir rejani koʻrsatkichlarini oʻqish uchun qayta hisoblash. Natija
jadvali yuqoridagi toʻr jadvali bilan bir xil ustunlarga ega.

In [ ]:
# @title Kod: fig-optimizer
#| label: fig-optimizer
#| fig-cap: "The optimizer's plans (blue) on top of the 441 grid plans. It finds the same curve and fills the gaps, because it is not tied to the grid's steps."
_ = plot_tradeoff(table, found=found)

In [ ]:
# @title Kod: optimizer-howto
#| label: optimizer-howto
#| echo: false
distinct = len(front_of(found))
grid_best = front["worst_month"].min()
say(
    f"How to read it: the blue dots are the {len(found)} plans the optimizer finished with, {distinct} of them "
    f"different. Most sit on or just beside the black staircase, and several lie between its steps: the optimizer may "
    f"choose any trigger and any ration, not only the grid's steps of 2 Mm³ and 5 %. A few blue dots reach further "
    f"right than the last black one: the optimizer also found plans with a worst month below {grid_best:.0f} %, at a "
    f"high price in water. It did this with 1 600 runs and without ever seeing the grid.",
    f"Buni qanday oʻqish: koʻk nuqtalar — optimallashtiruvchi yakunlagan {len(found)} reja, ulardan {distinct} tasi "
    f"har xil. Koʻpchiligi qora zinapoyaning ustida yoki yonida, bir nechtasi esa uning pogʻonalari orasida: "
    f"optimallashtiruvchi istalgan chegara va istalgan cheklashni tanlashi mumkin, toʻrning 2 mln m³ va 5 % lik "
    f"qadamlarini emas. Bir nechta koʻk nuqta oxirgi qora nuqtadan oʻngroqqa yetadi: optimallashtiruvchi eng yomon "
    f"oyi {grid_best:.0f} % dan past rejalarni ham topdi, suv boʻyicha qimmat narxda. U buni 1 600 ta hisob bilan va "
    f"toʻrni umuman koʻrmasdan qildi.",
    f"Как это читать: синие точки — {len(found)} планов, с которыми оптимизатор закончил, {distinct} из них разные. "
    f"Большинство лежат на чёрной лестнице или рядом с ней, а несколько — между её ступенями: оптимизатор может "
    f"выбрать любой порог и любую долю подачи, а не только шаги сетки в 2 млн м³ и 5 %. Несколько синих точек "
    f"уходят правее последней чёрной: оптимизатор нашёл и планы с худшим месяцем ниже {grid_best:.0f} %, по высокой "
    f"цене в воде. Он сделал это за 1 600 прогонов, ни разу не увидев сетку.",
)

# Nimaga eʼtibor berish kerak

In [ ]:
# @title Kod: notice
#| label: notice
#| echo: false
first, last = front.iloc[0], front.iloc[-1]
knee = front.loc[(front["worst_month"] - 50.0).abs().idxmin()]
beaten = int((~table["unbeaten"]).sum())
best_volume = table.loc[table["mean_shortage"].idxmin()]
say(
    f"**1. What you count decides the answer.** If the only goal is to deliver as much water as possible, there is "
    f"one best plan and it is *no rationing* (average shortage {best_volume['mean_shortage']:.2f} Mm³ per year, the "
    f"lowest of all 441 plans). The trade-off only appears once you also care about the worst month. Choosing the "
    f"goals is the most important modelling decision, and it is not a technical one.\n\n"
    f"**2. The curve bends.** Going from {first['worst_month']:.0f} % to {knee['worst_month']:.0f} % short in the "
    f"worst month costs {knee['mean_shortage'] - first['mean_shortage']:.1f} Mm³ per year. Going on from "
    f"{knee['worst_month']:.0f} % to {last['worst_month']:.0f} % costs another "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} Mm³ per year. The first steps are cheap, the last ones "
    f"very expensive. A plan near the bend is the natural place to start the discussion.\n\n"
    f"**3. Being short more often is part of the price.** Along the curve the number of short years rises from "
    f"{first['years_short']:.0f} to {last['years_short']:.0f} of 100.\n\n"
    f"**4. Most plans are simply bad.** {beaten} of the 441 plans are beaten, including the reasonable-sounding "
    f"\"ration lightly\". Intuition is a poor guide here; the search is not.",
    f"**1. Nimani hisoblashingiz javobni belgilaydi.** Agar yagona maqsad iloji boricha koʻp suv yetkazish boʻlsa, "
    f"bitta eng yaxshi reja bor va bu — *cheklash yoʻq* (oʻrtacha taqchillik yiliga "
    f"{best_volume['mean_shortage']:.2f} mln m³, barcha 441 reja ichida eng kami). Murosa faqat eng yomon oy ham "
    f"sizga muhim boʻlgandagina paydo boʻladi. Maqsadlarni tanlash — modellashtirishdagi eng muhim qaror, va u "
    f"texnik qaror emas.\n\n"
    f"**2. Egri chiziq bukiladi.** Eng yomon oyda {first['worst_month']:.0f} % dan {knee['worst_month']:.0f} % "
    f"taqchillikka oʻtish yiliga {knee['mean_shortage'] - first['mean_shortage']:.1f} mln m³ ga tushadi. "
    f"{knee['worst_month']:.0f} % dan {last['worst_month']:.0f} % gacha davom etish yana yiliga "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} mln m³ ga tushadi. Birinchi qadamlar arzon, oxirgilari "
    f"juda qimmat. Muhokamani bukilish yaqinidagi rejadan boshlash tabiiy.\n\n"
    f"**3. Koʻproq taqchillik koʻrish — bahoning bir qismi.** Egri chiziq boʻylab taqchil yillar soni 100 dan "
    f"{first['years_short']:.0f} tadan {last['years_short']:.0f} tagacha oshadi.\n\n"
    f"**4. Koʻpchilik rejalar shunchaki yomon.** 441 rejadan {beaten} tasi ustun kelingan, shu jumladan oqilona "
    f"tuyulgan \"ozgina cheklash\" ham. Bu yerda sezgi yomon yoʻlboshchi; qidiruv esa yaxshi.",
    f"**1. Ответ определяет то, что вы считаете.** Если единственная цель — подать как можно больше воды, есть один "
    f"лучший план, и это *без лимитирования* (средний дефицит {best_volume['mean_shortage']:.2f} млн м³ в год, "
    f"наименьший из всех 441 планов). Компромисс появляется только тогда, когда вам важен и худший месяц. Выбор "
    f"целей — самое важное решение при моделировании, и оно не техническое.\n\n"
    f"**2. Кривая изгибается.** Переход от {first['worst_month']:.0f} % к {knee['worst_month']:.0f} % дефицита в "
    f"худшем месяце стоит {knee['mean_shortage'] - first['mean_shortage']:.1f} млн м³ в год. Дальнейший переход от "
    f"{knee['worst_month']:.0f} % к {last['worst_month']:.0f} % стоит ещё "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} млн м³ в год. Первые шаги дёшевы, последние очень дороги. "
    f"План возле изгиба — естественная отправная точка для обсуждения.\n\n"
    f"**3. Более частый дефицит — часть цены.** Вдоль кривой число дефицитных лет растёт с "
    f"{first['years_short']:.0f} до {last['years_short']:.0f} из 100.\n\n"
    f"**4. Большинство планов просто плохи.** {beaten} из 441 планов превзойдены, включая разумное на слух «лёгкое "
    f"лимитирование». Интуиция здесь плохой советчик; поиск — хороший.",
)

# Yana bir blok: bugʻlanish

Hozirgacha omborda saqlangan suv u yerda tekin kutib turardi. Issiq iqlimda bunday emas: uning bir qismi
bugʻlanadi. TaqSimda bu — suv omboriga ulanadigan yana bitta kichik klass, *yoʻqotish qoidasi*:

In [ ]:
# @title Kod: evaporation-source
#| label: evaporation-source
#| echo: false
show_source(SummerEvaporation)

`calculate(...)` ni TaqSim har oy bir marta, oqim saqlangandan keyin va chiqarishdan oldin chaqiradi. U nima
yoʻqolgani va nima sababdan yoʻqolganini qaytaradi; TaqSim buni suv hajmidan ayiradi va `WaterLost` hodisasini
yozadi.

In [ ]:
# @title Kod: fig-evaporation
#| label: fig-evaporation
#| fig-cap: "The unbeaten plans without and with evaporation. With evaporation every plan delivers less, and a mild worst month becomes much more expensive."
# Same 441 plans, same river, but now the reservoir loses 3 % of its content in each summer month
with_evaporation = score_plans(inflows, plan_grid(), evaporation_share=EVAPORATION_SHARE_EXAMPLE)

# Draw the unbeaten plans of both tables on one chart
_ = plot_front_shift({"no evaporation": table, "3 % of the stored water evaporates each summer month": with_evaporation})

In [ ]:
# @title Kod: evaporation-text
#| label: evaporation-text
#| echo: false
front_e = front_of(with_evaporation)
at_50 = front[front["worst_month"] <= 50.0].iloc[0]
at_50_e = front_e[front_e["worst_month"] <= 50.0].iloc[0]
say(
    f"The two staircases are the unbeaten plans without (blue) and with (orange) evaporation. The orange one lies to "
    f"the right of and above the blue one: every plan delivers less, and the same worst month costs more water. It "
    f"also stops higher: with evaporation no plan gets the worst month below {front_e['worst_month'].min():.0f} %, "
    f"against {front['worst_month'].min():.0f} % without. "
    f"The 3 % is an illustrative figure, not a measurement. The effect is large: holding the worst month to 50 % "
    f"short costs {at_50['mean_shortage']:.1f} Mm³ per year without evaporation and "
    f"{at_50_e['mean_shortage']:.1f} Mm³ per year with it. Saving water for later is worth less when the saved "
    f"water does not keep.",
    f"Ikkita zinapoya — bugʻlanishsiz (koʻk) va bugʻlanish bilan (toʻq sariq) yengilmagan rejalar. Toʻq sariq "
    f"zinapoya koʻkning oʻngida va yuqorisida: har bir reja kamroq yetkazadi, va bir xil eng yomon oy koʻproq suvga "
    f"tushadi. U yuqoriroqda ham toʻxtaydi: bugʻlanish bilan hech bir reja eng yomon oyni "
    f"{front_e['worst_month'].min():.0f} % dan pastga tushira olmaydi, bugʻlanishsiz esa "
    f"{front['worst_month'].min():.0f} % gacha. "
    f"3 % — oʻlchov emas, tushuntirish uchun olingan raqam. Taʼsiri katta: eng yomon oyni 50 % taqchillikda "
    f"ushlab turish bugʻlanishsiz yiliga {at_50['mean_shortage']:.1f} mln m³, bugʻlanish bilan esa yiliga "
    f"{at_50_e['mean_shortage']:.1f} mln m³ ga tushadi. Saqlangan suv saqlanmasa, uni keyinga asrash kamroq "
    f"qadrli.",
    f"Две лестницы — непревзойдённые планы без испарения (синяя) и с испарением (оранжевая). Оранжевая лежит правее "
    f"и выше синей: каждый план подаёт меньше, а тот же худший месяц стоит больше воды. Она и обрывается выше: с "
    f"испарением ни один план не опускает худший месяц ниже {front_e['worst_month'].min():.0f} %, без испарения — "
    f"до {front['worst_month'].min():.0f} %. "
    f"3 % — иллюстративное число, не измерение. Эффект велик: удержать худший месяц на уровне 50 % дефицита стоит "
    f"{at_50['mean_shortage']:.1f} млн м³ в год без испарения и {at_50_e['mean_shortage']:.1f} млн м³ в год с "
    f"ним. Беречь воду на потом менее выгодно, когда сбережённая вода не сохраняется.",
)

# Yana bir isteʼmolchi: qishki dalalar

Ombor qishda toʻladi, Zarafshon vohasida esa qish boʻsh oʻtmaydi. Kuzgi bugʻdoy sugʻoriladi, bahorgi ekishdan
oldin shoʻrlangan dalalar yuviladi: tuz ildizlardan pastga yuvilib ketishi uchun dalaga suv bostiriladi. Ikkalasi
ham xuddi oʻsha ombordan, u toʻlishi kerak boʻlgan oylarda suv oladi. Bu isteʼmolchini beshinchi blok sifatida,
xoʻjalikdan pastda, oʻsha kanalda qoʻshamiz: **qishki dalalar**. Ular har qish oyida 2,5 mln m³, bir qishda 15 mln
m³ soʻraydi, yozda esa hech narsa soʻramaydi. Bu raqam, bugʻlanish ulushi kabi, tushuntirish uchun olingan.

In [ ]:
# @title Kod: fig-winter-network
#| label: fig-winter-network
#| fig-cap: "Five blocks. The winter fields sit on the canal below the farm: in winter they take what the reservoir releases for them, and whatever the reservoir spills."
_ = plot_network(winter=True)

Har qishda operator oldida yangi savol turadi. Qishki dalalarga berilgan suv yozgi erish qanchalik katta
boʻlishini hech kim bilmasidan oldin ketib boʻladi. Toʻliq bering — kuchsiz erish xoʻjalikni iyulda taqchil
qoldiradi. Ushlab turing — koʻp qishlarda bu suv kerak boʻlmagan boʻlib chiqadi. Qoida qish uchun yana ikki dastak
oladi:

- **Qishki chegara** (mln m³): *koʻrinib turgan suv* bundan kam boʻlsa, qishki dalalar cheklanadi. Hozircha
  koʻrinib turgan suv — bu shunchaki ombordagi suv.
- **Qishki ulush** (0 dan 1 gacha): qishki dalalar oʻshanda oladigan talab ulushi.

Yozgi dastaklar avvalgidek qoladi. Daftarning qolgan qismida biz ularni "keskin kesish" (chegara 16 mln m³, ulush
0,5), yaʼni ikkinchi hisobdagi rejada qotirib qoʻyamiz, shunda faqat qishki qaror oʻzgaradi.

In [ ]:
# @title Kod: winter-rule-source
#| label: winter-rule-source
#| echo: false
show_source(WinterRelease)

In [ ]:
# @title Kod: winter-two-runs
#| label: winter-two-runs
# Summer "cut deep" in both; the winter fields served in full, or given nothing when the store is below 24 Mm³
winter_in_full = simulate_winter(inflows, WINTER_IN_FULL)
hold_back = simulate_winter(inflows, WinterPlan(trigger=24.0, ration=0.0))
for name, run in {"Winter fields served in full": winter_in_full, "Hold back below 24 Mm³": hold_back}.items():
    print(
        f"{name:30s} farm short {run.scores.summer_shortage:.1f} Mm³/yr, "
        f"winter fields short {run.scores.winter_shortage:.1f} Mm³/yr"
    )

In [ ]:
# @title Kod: winter-two-runs-text
#| label: winter-two-runs-text
#| echo: false
full, back = winter_in_full.scores, hold_back.scores
say(
    f"Served in full, the winter fields get almost all of their 15 Mm³, and the farm's shortage rises from "
    f"{cut_deep.scores.mean_shortage:.1f} Mm³ a year (the same summer plan without the winter fields) to "
    f"{full.summer_shortage:.1f}. Holding back below 24 Mm³ gives the farm back most of that, "
    f"{full.summer_shortage - back.summer_shortage:.1f} Mm³ a year, at a price of "
    f"{back.winter_shortage - full.winter_shortage:.1f} Mm³ a year for the winter fields. Even \"nothing\" is not "
    f"quite nothing: in wet winters the reservoir spills, and the spilled water reaches the fields anyway.\n\n"
    f"The two goals are now the two users: the water the farm did not get in summer, and the water the winter "
    f"fields did not get in winter, both in Mm³ a year, both less is better. As before, we try every winter plan: "
    f"triggers from 0 to 40 Mm³ in steps of 1, rations in steps of 0.05, 861 plans.",
    f"Toʻliq berilganda qishki dalalar oʻzining 15 mln m³ ining deyarli hammasini oladi, xoʻjalikning taqchilligi "
    f"esa yiliga {cut_deep.scores.mean_shortage:.1f} mln m³ dan (qishki dalalarsiz oʻsha yozgi reja) "
    f"{full.summer_shortage:.1f} ga koʻtariladi. 24 mln m³ dan pastda ushlab turish xoʻjalikka buning koʻp qismini, "
    f"yiliga {full.summer_shortage - back.summer_shortage:.1f} mln m³ ni qaytaradi, qishki dalalar uchun esa yiliga "
    f"{back.winter_shortage - full.winter_shortage:.1f} mln m³ ga tushadi. Hatto \"hech narsa\" ham toʻliq hech "
    f"narsa emas: sersuv qishlarda ombor toshadi va toshgan suv baribir dalalarga yetib boradi.\n\n"
    f"Endi ikki maqsad — ikki isteʼmolchi: xoʻjalik yozda olmagan suv va qishki dalalar qishda olmagan suv, "
    f"ikkalasi ham yiliga mln m³ da, ikkalasida ham kam boʻlgani yaxshi. Avvalgidek, har bir qishki rejani sinab "
    f"koʻramiz: chegaralar 0 dan 40 mln m³ gacha 1 qadam bilan, ulushlar 0,05 qadam bilan, 861 ta reja.",
    f"При полной подаче зимние поля получают почти все свои 15 млн м³, а дефицит хозяйства растёт с "
    f"{cut_deep.scores.mean_shortage:.1f} млн м³ в год (тот же летний план без зимних полей) до "
    f"{full.summer_shortage:.1f}. Придерживание ниже 24 млн м³ возвращает хозяйству большую часть этого, "
    f"{full.summer_shortage - back.summer_shortage:.1f} млн м³ в год, ценой "
    f"{back.winter_shortage - full.winter_shortage:.1f} млн м³ в год для зимних полей. Даже «ничего» — не совсем "
    f"ничего: в многоводные зимы водохранилище сбрасывает излишек, и эта вода всё равно доходит до полей.\n\n"
    f"Две цели — теперь два водопользователя: вода, которую хозяйство не получило летом, и вода, которую зимние "
    f"поля не получили зимой, обе в млн м³ в год, обе — чем меньше, тем лучше. Как и раньше, перебираем все "
    f"зимние планы: пороги от 0 до 40 млн м³ с шагом 1, доли с шагом 0,05, 861 план.",
)

> **Qishki masala formulalarda.** Yozgi reja qotirilgan: $\theta^* = (16;\ 0{,}5)$; qishki reja —
> $\psi = (\tau_w, \rho_w)$. Qishki dalalar qish oyida $w_m = 2{,}5$ mln m³ soʻraydi, yozda hech narsa. Qish oyida
> qoida koʻrinib turgan suv $v_t$ ni qishki chegara bilan solishtiradi: hozircha faqat ombordagi suv, keyingi
> boʻlimda ombordagi suv plyus kelayotgan yozning $F_y$ prognozi. Avval xoʻjalikka beriladi, qolgani qishki
> dalalarga:
>
$$
r_t = \min\{\hat s_t,\; d_{m(t)} + \rho^{w}_t\, w_{m(t)}\}, \qquad
\rho^{w}_t = \begin{cases} \rho_w & \text{if } v_t < \tau_w \\ 1 & \text{otherwise,} \end{cases} \qquad
v_t = \hat s_t \;\text{ or }\; \hat s_t + F_{y(t)},
$$
>
> Yoz oylari yuqoridagi yozgi qoidaga boʻysunadi. Ikki koʻrsatkich — ikki isteʼmolchining taqchilligi, yiliga mln
> m³ da; masala avvalgi turda, $\tau_w \in [0, 40]$ (prognoz bilan $[0, 120]$) va $\rho_w \in [0, 1]$ boʻyicha:
>
$$
g_1(\psi) = \frac{1}{N}\sum_{t:\, m(t) \in S} \delta^{\text{farm}}_t, \qquad
g_2(\psi) = \frac{1}{N}\sum_{t:\, m(t) \notin S} \delta^{\text{fields}}_t, \qquad
\min_{\psi} \big(g_1(\psi),\, g_2(\psi)\big).
$$

In [ ]:
# @title Kod: fig-winter-front
#| label: fig-winter-front
#| fig-cap: "The unbeaten winter plans when the winter decision looks at the storage alone. Left is better for the farm, down is better for the winter fields."
# 861 winter plans, summer fixed at "cut deep"; the water in sight is the storage, so triggers run to 40 Mm³
storage_only = score_winter_plans(inflows, winter_grid(n_trigger=41, trigger_max=CAPACITY_MM3))
_ = plot_winter_fronts({"winter decided on the storage alone": storage_only})

In [ ]:
# @title Kod: winter-front-text
#| label: winter-front-text
#| echo: false
at_2 = least_winter_shortage(storage_only, 2.0)
at_3 = least_winter_shortage(storage_only, 3.0)
say(
    f"The same bent curve as in summer. Its bottom-right end is \"serve in full\": the winter fields lack almost "
    f"nothing, the farm is short {full.summer_shortage:.1f} Mm³ a year. Moving up and left, each step gives the "
    f"farm back some water and takes more from the winter fields. To hold the farm's shortage to 3 Mm³ a year, the "
    f"winter fields must give up {at_3:.1f} Mm³ a year; to hold it to 2, {at_2:.1f}. The operator who only "
    f"watches the storage pays this price in every winter the reservoir is low, including the many that a strong "
    f"melt would have rescued.",
    f"Yozdagi kabi bukilgan egri chiziq. Uning pastki oʻng uchi — \"toʻliq berish\": qishki dalalarga deyarli hech "
    f"narsa yetishmaydi, xoʻjalik esa yiliga {full.summer_shortage:.1f} mln m³ taqchil. Yuqoriga va chapga "
    f"yurganda har bir qadam xoʻjalikka bir oz suv qaytaradi va qishki dalalardan koʻproq oladi. Xoʻjalik "
    f"taqchilligini yiliga 3 mln m³ da ushlab turish uchun qishki dalalar yiliga {at_3:.1f} mln m³ dan voz kechishi "
    f"kerak; 2 da ushlab turish uchun — {at_2:.1f}. Faqat ombordagi suvga qaraydigan operator bu narxni ombor kam "
    f"suvli boʻlgan har bir qishda toʻlaydi, shu jumladan kuchli erish qutqarib qoladigan koʻplab qishlarda ham.",
    f"Та же изогнутая кривая, что и летом. Её нижний правый конец — «подавать полностью»: зимним полям почти "
    f"ничего не не хватает, хозяйству не хватает {full.summer_shortage:.1f} млн м³ в год. Двигаясь вверх и влево, "
    f"каждый шаг возвращает хозяйству немного воды и отнимает больше у зимних полей. Чтобы удержать дефицит "
    f"хозяйства на 3 млн м³ в год, зимние поля должны отдать {at_3:.1f} млн м³ в год; чтобы удержать на 2 — "
    f"{at_2:.1f}. Оператор, который смотрит только на запас, платит эту цену каждую зиму, когда водохранилище "
    f"маловодно, включая те многие зимы, которые сильное таяние всё равно бы спасло.",
)

# Yoz prognozi, qaror qishda

Operator suvni yoz nomaʼlum boʻlgani uchun ushlab turadi. Lekin u butunlay nomaʼlum emas. Qish oxiriga kelib yozda
eriydigan qorning koʻp qismi yogʻib boʻlgan boʻladi, va Markaziy Osiyo gidrometeorologiya xizmatlari har bahorda
oʻz daryolarining aprel–sentyabr oqimini qor qoplami, qishki yogʻin va oʻtgan oylar oqimi asosida prognoz qiladi
(Apel va boshq., 2018). Faraz qilaylik, bizning operatorda har qish boshida shunday prognoz bor: bitta raqam,
kelayotgan yozning oqimi. Aslida u qish oxirida, qor yogʻib boʻlgach keladi; bu yerda operator uni qish boshida
oladi — bu eng yaxshi holat, kechroq kelgan prognoz qanchaga arzishini esa 6-mashq soʻraydi.

Qoidaga bitta oʻzgartirish kerak. Koʻrinib turgan suv endi faqat ombordagi suv emas, balki ombordagi suv **plyus
prognoz**, va qishki chegara shu bilan solishtiriladi. Boshqa hech narsa oʻzgarmaydi: oʻsha dastaklar, oʻsha yozgi
reja, oʻsha daryo. Koʻrinib turgan suvga endi bir yoz ham kirgani uchun chegara 40 emas, 120 mln m³ gacha boradi.

Eng yaxshi holatdan boshlaymiz: hamisha toʻgʻri chiqadigan prognoz. U bizga bu yerda prognoz eng koʻpi bilan
qanchalik qadrli boʻlishi mumkinligini koʻrsatadi.

In [ ]:
# @title Kod: fig-forecast-front
#| label: fig-forecast-front
#| fig-cap: "The unbeaten winter plans when the winter decision looks at the storage alone (orange) and at the storage plus a perfect forecast of the coming summer (blue)."
perfect = seasonal_forecast(inflows, skill=1.0)  # one number per year: the coming summer's inflow, exactly
with_perfect = score_winter_plans(inflows, winter_grid(n_trigger=41), forecast=perfect)  # 861 plans, triggers to 120
_ = plot_winter_fronts({"storage plus a perfect forecast": with_perfect, "storage alone": storage_only})

In [ ]:
# @title Kod: fig-winter-decisions
#| label: fig-winter-decisions
#| fig-cap: "Two winter plans with the same summer shortage, 2 Mm³ a year at most, in the twenty years around the worst drought. Top: what the winter fields received each winter. Bottom: the summer that followed."
# From each curve, the plan that holds the farm's shortage to 2 Mm³ a year with the least loss for the winter fields
chosen = {
    "storage plus a perfect forecast": winter_plan_within(with_perfect, 2.0),
    "storage alone": winter_plan_within(storage_only, 2.0),
}
runs = {
    label: simulate_winter(inflows, winter, forecast=perfect if "forecast" in label else None)
    for label, winter in chosen.items()
}
_ = plot_winter_decisions(runs, inflows, first_year=first_year)

In [ ]:
# @title Kod: forecast-front-text
#| label: forecast-front-text
#| echo: false
at_2_perfect = least_winter_shortage(with_perfect, 2.0)
at_3_perfect = least_winter_shortage(with_perfect, 3.0)
gain_2 = (at_2 - at_2_perfect) / at_2 * 100.0
say(
    f"- **The blue curve lies below the orange one.** At every summer shortage the farm accepts, the winter fields "
    f"lose less. Holding the farm to 2 Mm³ a year costs the winter fields {at_2:.1f} Mm³ a year without the "
    f"forecast and {at_2_perfect:.1f} with it, {gain_2:.0f} % less; at 3 Mm³ a year, {at_3:.1f} against "
    f"{at_3_perfect:.1f}.\n"
    f"- **Why.** Look at the two plans in the second figure. The plan without the forecast holds back whenever the "
    f"reservoir is low, which is mostly the winter *after* a dry summer. The plan with the forecast holds back in "
    f"the winters *before* a weak summer and serves in full before a strong one. It rations fewer winters, and the "
    f"right ones.\n"
    f"- **This gap is the ceiling.** No real forecast can do better than one that is always right. Whatever a "
    f"forecast service can offer is worth at most this much here: {at_2 - at_2_perfect:.1f} Mm³ a year for the "
    f"winter fields at a summer shortage of 2. In a system where more decisions are taken before the melt, the "
    f"ceiling is higher.",
    f"- **Koʻk egri chiziq toʻq sarigʻidan pastda yotadi.** Xoʻjalik rozi boʻlgan har bir yozgi taqchillikda qishki "
    f"dalalar kamroq yoʻqotadi. Xoʻjalikni yiliga 2 mln m³ da ushlab turish qishki dalalarga prognozsiz yiliga "
    f"{at_2:.1f} mln m³, prognoz bilan {at_2_perfect:.1f} ga tushadi, yaʼni {gain_2:.0f} % kam; yiliga 3 mln m³ "
    f"da — {at_3:.1f} ga qarshi {at_3_perfect:.1f}.\n"
    f"- **Nima uchun.** Ikkinchi rasmdagi ikki rejaga qarang. Prognozsiz reja ombor kam suvli boʻlganda ushlab "
    f"turadi, bu esa asosan quruq yozdan *keyingi* qish. Prognozli reja kuchsiz yozdan *oldingi* qishlarda ushlab "
    f"turadi va kuchli yozdan oldin toʻliq beradi. U kamroq qishni cheklaydi, va aynan keraklilarini.\n"
    f"- **Bu farq — shift.** Hech bir haqiqiy prognoz hamisha toʻgʻri chiqadiganidan yaxshiroq boʻla olmaydi. "
    f"Prognoz xizmati taklif qila oladigan narsa bu yerda eng koʻpi bilan shuncha turadi: yozgi taqchillik 2 "
    f"boʻlganda qishki dalalar uchun yiliga {at_2 - at_2_perfect:.1f} mln m³. Erishdan oldin koʻproq qaror "
    f"qabul qilinadigan tizimda shift balandroq.",
    f"- **Синяя кривая лежит ниже оранжевой.** При любом летнем дефиците, на который соглашается хозяйство, зимние "
    f"поля теряют меньше. Удержать хозяйство на 2 млн м³ в год стоит зимним полям {at_2:.1f} млн м³ в год без "
    f"прогноза и {at_2_perfect:.1f} с ним, на {gain_2:.0f} % меньше; при 3 млн м³ в год — {at_3:.1f} против "
    f"{at_3_perfect:.1f}.\n"
    f"- **Почему.** Посмотрите на два плана на второй рисунке. План без прогноза придерживает воду всякий раз, "
    f"когда водохранилище маловодно, а это в основном зима *после* сухого лета. План с прогнозом придерживает "
    f"воду в зимы *перед* слабым летом и подаёт полностью перед сильным. Он лимитирует меньше зим, и именно те, "
    f"что нужно.\n"
    f"- **Этот разрыв — потолок.** Ни один реальный прогноз не может быть лучше того, что всегда верен. Что бы ни "
    f"предложила прогностическая служба, здесь это стоит самое большее столько: {at_2 - at_2_perfect:.1f} млн м³ в "
    f"год для зимних полей при летнем дефиците 2. В системе, где больше решений принимается до таяния, потолок "
    f"выше.",
)

# Xato qilishi mumkin boʻlgan prognoz

Hech bir prognoz hamisha toʻgʻri emas. Uning qanchalik yaxshi ekanini bitta raqam bilan ifodalaymiz — **sifati**:
prognozlar haqiqatda kelgan yozlarga qanchalik yaqin ergashadi, 0 dan (hech qanday bogʻliqlik yoʻq) 1 gacha
(hamisha toʻgʻri). Texnik nomi — korrelyatsiya koeffitsiyenti. Sifati 0 boʻlgan prognozchi har yili "oʻrtacha
yoz" deyishdan yaxshiroq hech narsa qila olmaydi.

`seasonal_forecast` shunday prognozlar yasaydi: prognozchi kelayotgan yozni berilgan sifat bilan, shovqin orqali
koʻradi va koʻrganiga koʻra eng yaxshi taxminni beradi. Oʻrtacha olganda taxmin na haddan tashqari dadil, na
haddan tashqari ehtiyotkor — statistik prognoz kabi. Ikkita misol:

> **Prognoz formulalarda.** $y$ yilning yozgi oqimi $Q_y = e^{\mu + \sigma z_y}$, bunda $z_y$ — standart baho
> (daryo lognormal). Prognozchi kelayotgan yozni $\varepsilon_y$ shovqin orqali koʻradi va koʻrganiga koʻra eng
> yaxshi taxminni beradi:
>
$$
\tilde z_y = \kappa\, z_y + \sqrt{1-\kappa^2}\; \varepsilon_y, \qquad
F_y = \mathbb{E}\big[Q_y \mid \tilde z_y\big] = \exp\!\Big(\mu + \sigma \kappa \tilde z_y + \tfrac{1}{2}\sigma^2 (1-\kappa^2)\Big),
$$
>
> shunda u koʻrgan narsa bilan haqiqat orasidagi korrelyatsiya — $\kappa$ sifat, prognoz $\kappa = 0$ da uzoq
> muddatli oʻrtacha, $\kappa = 1$ da yozning oʻzi. Prognozning qadri qotirilgan yozgi taqchillik $\bar c$ da
> (quyida yiliga 2 mln m³) oʻlchanadi: xoʻjalikka hech narsaga tushmasdan u tejab qolgan qishki suv,
>
$$
V(\kappa) = g_2^{*}(\text{no forecast}) - g_2^{*}(\kappa), \qquad
g_2^{*}(\cdot) = \min\big\{\, g_2(\psi) : g_1(\psi) \le \bar c \,\big\}.
$$
>
> $V(1)$, mukammal prognozning qadri — shift.

In [ ]:
# @title Kod: fig-forecast-skill
#| label: fig-forecast-skill
#| fig-cap: "One hundred summers and their forecasts, for a forecast of skill 0.5 and one of skill 0.9. On the grey line the forecast was exactly right."
_ = plot_forecast_skill(inflows, skills=(0.5, 0.9))

In [ ]:
# @title Kod: fig-skill-fronts
#| label: fig-skill-fronts
#| fig-cap: "The unbeaten winter plans for forecasts of growing skill. The curves move down as the skill grows, from storage alone to the perfect forecast."
# Six more forecasts of growing skill, 861 winter plans each (this cell runs for a few minutes)
tables = {1.0: with_perfect, **score_by_skill(inflows, (0.0, 0.3, 0.5, 0.65, 0.8, 0.9), winter_grid(n_trigger=41))}
_ = plot_winter_fronts(
    {
        "perfect forecast": with_perfect,
        "forecast with skill 0.8": tables[0.8],
        "forecast with skill 0.5": tables[0.5],
        "storage alone": storage_only,
    }
)

In [ ]:
# @title Kod: fig-forecast-value
#| label: fig-forecast-value
#| fig-cap: "What the forecast is worth against its skill: the winter shortage left when the farm's shortage is held to 2 Mm³ a year. The orange line is the operator without a forecast."
curve = forecast_value(tables, summer_cap=2.0)
_ = plot_forecast_value(curve, no_forecast=at_2, summer_cap=2.0)

In [ ]:
# @title Kod: forecast-value-text
#| label: forecast-value-text
#| echo: false
value_at = dict(zip(curve["skill"], curve["winter_shortage"]))
half_way = (at_2 - value_at[0.5]) / (at_2 - value_at[1.0]) * 100.0
say(
    f"- **A forecast with no skill is no forecast.** At skill 0 the forecast says 40 Mm³ every year, and the rule is "
    f"back to watching the storage: the curve starts on the orange line, within the grid's steps "
    f"({value_at[0.0]:.1f} against {at_2:.1f} Mm³ a year).\n"
    f"- **Value grows with skill, but not in proportion.** A forecast of skill 0.5, halfway to perfect, delivers "
    f"{half_way:.0f} % of what the perfect forecast delivers; one of skill 0.8 leaves the winter fields "
    f"{value_at[0.8]:.1f} Mm³ a year short, against {value_at[1.0]:.1f} for the perfect forecast and {at_2:.1f} "
    f"with none. Most of the value sits in the last stretch of skill. The curve wiggles because the forecast "
    f"errors are one random draw of a hundred years; the trend is what counts.\n"
    f"- **Where real forecasts sit.** The statistical forecasts that Apel et al. (2018) built for rivers of Central "
    f"Asia explain 60 to 80 % of the year-to-year variation of the April–September flow when issued in January, "
    f"and 68 to 97 % when issued on 1 April, the date that matters most for the region's water planning. In our "
    f"terms that is a skill of roughly 0.8 to 0.95: real forecasts sit on the right-hand part of the curve, where "
    f"the value is.\n"
    f"- **The forecast's worth depends on the decision, not only on the forecast.** The same forecast was worth "
    f"nothing for the summer decisions of this reservoir (we tried: the operator learns the summer by May, and "
    f"water saved in one month is still there the next), and worth {at_2 - value_at[1.0]:.1f} Mm³ a year for the "
    f"winter decision. A forecast is worth what the decision it informs can do with it, a point made for weather "
    f"forecasts by Murphy (1993) and for reservoirs by Anghileri et al. (2016).\n"
    f"- **What the operator does with a wrong forecast matters.** Here the rule takes the forecast at its word, and "
    f"the optimizer compensates by choosing the trigger. A cautious operator could count only part of the "
    f"forecast; exercise 5 asks whether that helps.",
    f"- **Sifati yoʻq prognoz — prognoz emas.** Sifat 0 da prognoz har yili 40 mln m³ deydi, va qoida yana "
    f"ombordagi suvga qarashga qaytadi: egri chiziq toʻr qadamlari doirasida toʻq sariq chiziqdan boshlanadi "
    f"(yiliga {value_at[0.0]:.1f} ga qarshi {at_2:.1f} mln m³).\n"
    f"- **Qadr sifat bilan oʻsadi, lekin mutanosib emas.** Sifati 0,5 boʻlgan, mukammalga yarim yoʻl bosgan "
    f"prognoz mukammal prognoz beradiganning {half_way:.0f} % ini beradi; sifati 0,8 boʻlgani qishki dalalarni "
    f"yiliga {value_at[0.8]:.1f} mln m³ taqchil qoldiradi, mukammal prognozda {value_at[1.0]:.1f}, prognozsiz "
    f"{at_2:.1f}. Qadrning koʻp qismi sifatning oxirgi boʻlagida yotadi. Egri chiziq tebranadi, chunki prognoz "
    f"xatolari — yuz yillik bitta tasodifiy tanlanma; muhimi — umumiy yoʻnalish.\n"
    f"- **Haqiqiy prognozlar qayerda turadi.** Apel va boshq. (2018) Markaziy Osiyo daryolari uchun qurgan statistik "
    f"prognozlar aprel–sentyabr oqimining yildan yilga oʻzgarishining 60–80 % ini yanvarda berilganda, 68–97 % ini "
    f"esa 1 aprelda — mintaqa suv rejalashtiruvi uchun eng muhim sanada — berilganda tushuntiradi. Bizning "
    f"atamalarda bu taxminan 0,8 dan 0,95 gacha sifat: haqiqiy prognozlar egri chiziqning oʻng qismida, qadr "
    f"yotgan joyda turadi.\n"
    f"- **Prognozning qadri faqat prognozga emas, qarorga bogʻliq.** Xuddi shu prognoz bu omborning yozgi "
    f"qarorlari uchun hech narsaga arzimadi (biz sinab koʻrdik: operator yozni may oyigacha bilib oladi, bir oyda "
    f"saqlangan suv esa keyingi oyda ham joyida turadi), qishki qaror uchun esa yiliga "
    f"{at_2 - value_at[1.0]:.1f} mln m³ ga arzidi. Prognoz u xabardor qilgan qaror u bilan nima qila olsa, "
    f"shunchaga arziydi — bu fikrni ob-havo prognozlari uchun Murphy (1993), suv omborlari uchun Anghileri va "
    f"boshq. (2016) aytgan.\n"
    f"- **Operator notoʻgʻri prognoz bilan nima qilishi muhim.** Bu yerda qoida prognozga soʻzsiz ishonadi, "
    f"optimallashtirgich esa chegarani tanlab buni qoplaydi. Ehtiyotkor operator prognozning faqat bir qismini "
    f"hisobga olishi mumkin edi; 5-mashq bu yordam beradimi, deb soʻraydi.",
    f"- **Прогноз без качества — не прогноз.** При качестве 0 прогноз каждый год говорит 40 млн м³, и правило снова "
    f"смотрит только на запас: кривая начинается на оранжевой линии, в пределах шага сетки "
    f"({value_at[0.0]:.1f} против {at_2:.1f} млн м³ в год).\n"
    f"- **Ценность растёт с качеством, но не пропорционально.** Прогноз качества 0,5, на полпути к идеальному, даёт "
    f"{half_way:.0f} % того, что даёт идеальный; прогноз качества 0,8 оставляет зимним полям дефицит "
    f"{value_at[0.8]:.1f} млн м³ в год против {value_at[1.0]:.1f} у идеального и {at_2:.1f} без прогноза. "
    f"Большая часть ценности лежит на последнем отрезке качества. Кривая колеблется, потому что ошибки прогноза — "
    f"одна случайная выборка из ста лет; важна тенденция.\n"
    f"- **Где находятся настоящие прогнозы.** Статистические прогнозы, которые Apel et al. (2018) построили для рек "
    f"Центральной Азии, объясняют 60–80 % межгодовой изменчивости стока за апрель–сентябрь при выпуске в январе и "
    f"68–97 % при выпуске 1 апреля — в дату, которая важнее всего для водного планирования региона. В наших "
    f"терминах это качество примерно от 0,8 до 0,95: настоящие прогнозы находятся на правой части кривой, там, "
    f"где и лежит ценность.\n"
    f"- **Ценность прогноза зависит от решения, а не только от прогноза.** Тот же прогноз ничего не стоил для "
    f"летних решений этого водохранилища (мы проверяли: оператор узнаёт лето к маю, а вода, сбережённая в одном "
    f"месяце, никуда не девается к следующему) и стоил {at_2 - value_at[1.0]:.1f} млн м³ в год для зимнего "
    f"решения. Прогноз стоит столько, сколько может сделать с ним решение, которое он обслуживает, — мысль, "
    f"высказанная для прогнозов погоды Murphy (1993), а для водохранилищ — Anghileri et al. (2016).\n"
    f"- **Важно, что оператор делает с ошибочным прогнозом.** Здесь правило верит прогнозу на слово, а "
    f"оптимизатор компенсирует это выбором порога. Осторожный оператор мог бы учитывать лишь часть прогноза; "
    f"упражнение 5 спрашивает, помогает ли это.",
)

# Boshqa daryo: maqoladagi rejim

Maqoladagi daryo qishda toʻlib oqadi va yozda eng kam suvli — biznikining aksi. Bir xil ombor, bir xil xoʻjalik,
bir xil yuzta tasodifiy qiymat; faqat fasllar oʻrin almashgan. Murosa bir xil koʻrinadimi?

In [ ]:
# @title Kod: fig-regimes
#| label: fig-regimes
#| fig-cap: "Left: the average month of both rivers against the farm's need. Right: the unbeaten plans of both. Same reservoir, same need, same random seed."
paper_inflows = synthetic_inflows(regime=PAPER_REGIME)  # the paper's seasons: winter 40 Mm³, summer 25 Mm³
paper_table = score_plans(paper_inflows, plan_grid())  # the same 441 plans on that river

_ = plot_regime_comparison(
    {"Central Asian regime (this notebook)": (inflows, table), "Paper regime (winter peak)": (paper_inflows, paper_table)}
)

In [ ]:
# @title Kod: regimes-text
#| label: regimes-text
#| echo: false
paper_front = front_of(paper_table)
paper_base = simulate_plan(paper_inflows, NO_RATIONING).scores
ours_50 = front[front["worst_month"] <= 50.0].iloc[0]
paper_50 = paper_front[paper_front["worst_month"] <= 50.0].iloc[0]
say(
    f"- **How to read the figure.** Left: the average inflow of each month, blue for our river, orange for the "
    f"paper's; the black step is the farm's need, 0.83 Mm³ a month in winter and 7.5 in summer. Our summer bars "
    f"reach almost up to the need; the paper's summer bars stay far below it, so its reservoir has to carry most "
    f"of the summer water over from winter. Right: the unbeaten plans of both rivers, on the chart you know.\n"
    f"- **Same shape.** Both curves bend the same way: cheap at first, expensive at the end. The example works for "
    f"either river.\n"
    f"- **The paper's river is the harder case.** Without rationing it is {paper_base.worst_month:.0f} % short in "
    f"its worst month (ours: {s.worst_month:.0f} %), and holding the worst month to 50 % costs "
    f"{paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} Mm³ per year there against "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} here. With a summer peak the water arrives when the farm "
    f"needs it, and the reservoir only has to cover the gap.\n"
    f"- **What runs our reservoir dry** is a poor winter (little to store) followed by a weak melt. That is the "
    f"Central Asian pattern: the reservoir is filled from the low winter flow, as Kattakurgan is.",
    f"- **Rasmni qanday oʻqish.** Chapda: har bir oyning oʻrtacha oqimi, koʻk — bizning daryo, toʻq sariq — "
    f"maqoladagi daryo; qora pogʻona — xoʻjalik talabi, qishda oyiga 0,83 mln m³, yozda 7,5. Bizning yoz ustunlari "
    f"deyarli talabgacha yetadi; maqoladagi daryoning yoz ustunlari undan ancha pastda, shuning uchun uning ombori "
    f"yoz suvining koʻp qismini qishdan olib oʻtishi kerak. Oʻngda: ikkala daryoning yengilmagan rejalari, sizga "
    f"tanish diagrammada.\n"
    f"- **Shakli bir xil.** Ikkala egri chiziq ham bir xil bukiladi: avval arzon, oxirida qimmat. Misol ikkala daryo "
    f"uchun ham ishlaydi.\n"
    f"- **Maqoladagi daryo — qiyinroq holat.** Cheklashsiz uning eng yomon oyi {paper_base.worst_month:.0f} % "
    f"taqchil (bizniki: {s.worst_month:.0f} %), va eng yomon oyni 50 % da ushlab turish u yerda yiliga "
    f"{paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} mln m³, bizda esa "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} mln m³ ga tushadi. Yozgi toshqin bilan suv xoʻjalikka kerak "
    f"paytda keladi, va ombor faqat farqni qoplashi kerak.\n"
    f"- **Bizning omborni quritadigan narsa** — kam suvli qish (saqlashga kam suv) va undan keyingi kuchsiz erish. "
    f"Bu Markaziy Osiyo manzarasi: ombor, Kattaqoʻrgʻon suv ombori kabi, kam suvli qishki oqimdan toʻldiriladi.",
    f"- **Как читать рисунок.** Слева: средний приток каждого месяца, синий — наша река, оранжевый — река из "
    f"статьи; чёрная ступенька — потребность хозяйства, 0,83 млн м³ в месяц зимой и 7,5 летом. Наши летние столбики "
    f"почти дотягивают до потребности; летние столбики реки из статьи остаются далеко ниже, так что её "
    f"водохранилищу приходится переносить большую часть летней воды с зимы. Справа: непревзойдённые планы обеих "
    f"рек на уже знакомом графике.\n"
    f"- **Та же форма.** Обе кривые изгибаются одинаково: сначала дёшево, в конце дорого. Пример работает для обеих "
    f"рек.\n"
    f"- **Река из статьи — более трудный случай.** Без лимитирования её худший месяц имеет дефицит "
    f"{paper_base.worst_month:.0f} % (у нашей: {s.worst_month:.0f} %), а удержание худшего месяца на уровне 50 % "
    f"стоит там {paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} млн м³ в год против "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} у нас. При летнем пике вода приходит тогда, когда она нужна "
    f"хозяйству, и водохранилищу остаётся лишь покрыть разницу.\n"
    f"- **Наше водохранилище опустошает** скудная зима (мало что накопить), за которой следует слабое таяние. Это "
    f"центральноазиатская картина: водохранилище наполняется из скудного зимнего стока, как Каттакурганское.",
)

# Oʻyinchoq modeldan haqiqiy suv omboriga

- **Haqiqiy qoidalarda xuddi shu dastaklar, lekin koʻp marta.** Ushbu ombordagi Zarafshon modelidagi Kattaqoʻrgʻon
  suv omborining chiqarish qoidasida (`ZRBReleaseRule`) chiqarish kamaytiriladigan suv sathi (`v1`) va kamaytirish
  koeffitsiyenti (`buffer_coef`) bor: bizning chegara va ulush, har oy uchun alohida qiymat bilan. Shuning uchun
  toʻliq modelga optimallashtirgich kerak.
- **Bu oʻyinchoq model nimani hisobga olmaydi.** Qish davomida yangilanib boradigan prognoz va u bilan keladigan
  noaniqlik oraligʻi; turli ustuvorlikdagi yana koʻproq isteʼmolchi; ekinlar taqchillikka aslida qanday javob
  berishi; koʻl yuzasiga bogʻliq bugʻlanish; qaytuvchi oqimlar.
- **Bir daryo — bir daryo.** Bu yerdagi raqamlar yuz yillik shu bitta sunʼiy qatorga tegishli. Boshqa qatorlar
  boshqa raqamlarni beradi. Shakl ("cheklash yoʻq"dan "keskin kesish"gacha bukilgan egri chiziq) biz sinagan boshqa
  uchta qator uchun ham bir xil boʻldi.

# Mashqlar

1. **Kattaroq ombor.** `rationing_coursebook.py` da `CAPACITY_MM3 = 60` qilib qoʻying. Cheklash hali ham oʻzini
   oqlaydimi? Bukilish endi qayerda?
2. **Boshqa ikkinchi maqsad.** `objectives()` da "eng yomon oy"ni "taqchil yillar" bilan almashtiring va
   optimallashtirgichni qayta ishga tushiring. Endi qaysi rejalar ustun kelib boʻlmaydigan? "Cheklash yoʻq" hali ham
   ular orasidami?
3. **Boshqa daryo.** `synthetic_inflows(seed=1)`, soʻngra `synthetic_inflows(regime=PAPER_REGIME, seed=1)` dan
   foydalaning. Qaysi raqamlar oʻzgaradi, nima oʻzgarmaydi?
4. **Mavsumiy qoida.** Qoidaga ikkita ulush bering: biri yoz boshi, biri yoz oxiri uchun. Optimallashtirgich ikki
   dastakli egri chiziqdan ustun keladigan rejalarni topadimi?
5. **Ehtiyotkor operator.** `WinterRelease` da prognozning faqat bir qismini hisobga oling:
   `node.storage + 0.7 * forecast`. Sifat egri chizigʻini qayta hisoblang. Ehtiyotkorlik kuchsiz prognozda (sifat
   0,5) yordam beradimi? Kuchli prognozda zarar keltiradimi?
6. **Kech kelgan prognoz.** Haqiqiy prognozlar qish oxirida, qor yogʻib boʻlgach keladi. Qoida prognozdan faqat
   qishning oxirgi ikki oyida (`month >= 4`) foydalansin, undan oldin esa faqat ombordagi suvga qarasin.
   Prognoz qadrining qancha qismi saqlanib qoladi?

Colab da mashqni qanday bajarish: har bir katakning kodi yigʻilgan; "Show code" ni bosib uni koʻring va oʻzgartiring,
soʻng undan pastdagi kataklarni qayta ishga tushiring. 1-mashq faylni oʻzgartirishni soʻraydi, Colab bunga yoʻl
qoʻymaydi; uning oʻrniga toʻr boʻlimidan oldin yangi katakda shuni bajaring, soʻng oʻsha boʻlimni qayta ishga
tushiring:

```python
import zarafshan_taqsim.rationing_coursebook as m
m.CAPACITY_MM3 = 60.0
```

# Manbalar

- Hashimoto, T., Stedinger, J. R., Loucks, D. P. (1982). Reliability, resiliency, and vulnerability criteria for
  water resource system performance evaluation. *Water Resources Research* 18(1), 14–20. The reservoir, its
  numbers (p. 17, Table 1; seasons swapped here) and the trade-off between failing often and failing badly
  (Fig. 9).
- Draper, A. J., Lund, J. R. (2004). Optimal hedging and carryover storage value. *Journal of Water Resources
  Planning and Management* 130(1), 83–87. When rationing pays (p. 83) and why a rule with two parameters is often
  enough (p. 87).
- Apel, H., Abdykerimova, Z., Agalhanova, M., Baimaganbetov, A., Gavrilenko, N., Gerlitz, L., Kalashnikova, O.,
  Unger-Shayesteh, K., Vorogushyn, S., Gafurov, A. (2018). Statistical forecast of seasonal discharge in Central
  Asia using observational records: development of a generic linear modelling tool for operational water resource
  management. *Hydrology and Earth System Sciences* 22, 2225–2254. How the region's hydrometeorological services
  forecast the April–September flow, and from what.
- Anghileri, D., Voisin, N., Castelletti, A., Pianosi, F., Nijssen, B., Lettenmaier, D. P. (2016). Value of
  long-term streamflow forecasts to reservoir operations for water supply in snow-dominated river catchments.
  *Water Resources Research* 52, 4209–4225. Forecast value measured against the perfect forecast, for a reservoir.
- Murphy, A. H. (1993). What is a good forecast? An essay on the nature of goodness in weather forecasting.
  *Weather and Forecasting* 8, 281–293. The distinction between a forecast's quality and its value to a decision.
- Deb, K., Pratap, A., Agarwal, S., Meyarivan, T. (2002). A fast and elitist multiobjective genetic algorithm:
  NSGA-II. *IEEE Transactions on Evolutionary Computation* 6(2), 182–197. The search method behind
  `taqsim.optimize`.